# HP / TB roofline sweep for fpint_improve FFN kernel

see `analysis.md` for reuse-factor derivation.

Kernel reference: `tests/regression/fpint_gemm_ffn_hw_improve/kernel.cpp`
- DMA tile 128x128x128, MXU 32x32 microtile
- input FP16 (2B), weight INT4 (0.5B), scale/zp FP16 (2B), output FP16 (2B)

Architecture ceilings (F = core freq [GHz]):
- MXU peak  = 2 * 32 * 32 * F [GOps/s] (= 2048*F)
- HBM BW    = HP * 64 * F [GB/s]
- TMEM BW   = TB * 64 * F [GB/s]

Reuse pattern: per-HBM-load reuse = NT for input, MT for weight/scale
(NOT N/M; kernel re-loads input per nt_dma, weight/scale per mt).

In [ ]:
%matplotlib inline
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
sys.path.insert(0, os.path.join(ROOT, 'tools', 'roofline'))
from roofline import RooflineModel
import matplotlib.pyplot as plt

## 1. Architecture constants and OI helpers

In [ ]:
# ---- HW constants ----
F_GHZ       = 0.300          # core frequency [GHz] (Alveo U55C baseline)
MXU_DIM     = 32             # 32x32 systolic
PORT_WIDTH  = 64             # TMEM bank / HBM AXI port = 64 B

MXU_GOPS  = 2 * MXU_DIM * MXU_DIM * F_GHZ     # 2048 * F  [GOps/s]
def hbm_gbps(HP):  return HP * PORT_WIDTH * F_GHZ
def tmem_gbps(TB): return TB * PORT_WIDTH * F_GHZ

print(f'MXU peak    : {MXU_GOPS:.1f} GOps/s  (F={F_GHZ*1000:.0f} MHz)')
for hp in (1, 2, 4, 8):
    print(f'  HBM  HP={hp}  -> {hbm_gbps(hp):6.1f} GB/s, ridge={MXU_GOPS/hbm_gbps(hp):5.2f} FLOPs/B')
for tb in (1, 2, 4, 8):
    print(f'  TMEM TB={tb}  -> {tmem_gbps(tb):6.1f} GB/s, ridge={MXU_GOPS/tmem_gbps(tb):5.2f} FLOPs/B')

In [ ]:
# ---- Tile / element constants (fpint_improve FFN kernel) ----
DMA_MT, DMA_NT, DMA_KT = 128, 128, 128
MXU_KT, MXU_NT         = 32, 32

B_IN  = 2.0   # input  FP16
B_W   = 0.5   # weight INT4
B_Q   = 2.0   # scale/zp FP16
B_OUT = 2.0   # output FP16

def tile_flops(cm, cn, ck):
    return 2 * cm * cn * ck

def hbm_bytes_per_tile(cm, cn, ck, qblk):
    """HBM -> TMEM bytes for one (mt, nt_dma, kt) DMA tile (qdir=0 / QCOL)."""
    b_in   = cm * ck * B_IN
    b_w    = ck * cn * B_W
    b_sc   = (ck / qblk) * cn * B_Q
    b_zp   = (ck / qblk) * cn * B_Q
    return dict(input=b_in, weight=b_w, scale=b_sc, zp=b_zp,
                total=b_in+b_w+b_sc+b_zp)

def tmem_bytes_per_block(cm, cn, ck, qblk, k_tiles):
    """Per (mt, nt_dma) block: TMEM read+write traffic summed over k_tiles.
    nb_count / kb_count are computed from the tile shape (NOT hardcoded)."""
    nb_count = cn // MXU_NT
    kb_count = ck // MXU_KT
    hbm_fill = hbm_bytes_per_tile(cm, cn, ck, qblk)['total']    # DMA_LOAD -> TMEM write
    mxu_in   = nb_count * cm * ck * B_IN                        # LOAD_INPUT re-read per nb
    mxu_w    = ck * cn * B_W                                    # LOAD_WEIGHT: unique per tile
    mxu_qp   = nb_count * kb_count * 128                        # qparam per microtile
    out_wr   = cm * cn * B_OUT                                  # MXU_STORE -> TMEM (once per (mt,nt_dma))
    out_rd   = cm * cn * B_OUT                                  # DMA_STORE reads TMEM
    return k_tiles * (hbm_fill + mxu_in + mxu_w + mxu_qp) + out_wr + out_rd

def oi_hbm(cm, cn, ck, qblk, include_output=True, k_tiles=1):
    vols    = hbm_bytes_per_tile(cm, cn, ck, qblk)
    flops   = tile_flops(cm, cn, ck)
    total_b = k_tiles * vols['total'] + (cm * cn * B_OUT if include_output else 0)
    total_f = k_tiles * flops
    return total_f / total_b

def oi_tmem(cm, cn, ck, qblk, k_tiles):
    total_b = tmem_bytes_per_block(cm, cn, ck, qblk, k_tiles)
    total_f = k_tiles * tile_flops(cm, cn, ck)
    return total_f / total_b

def per_tensor_oi_hbm(cm, cn, ck, qblk):
    """(cm, cn, ck) = min(M, MT), min(N, NT), min(K, KT) — per-tile ceiling values."""
    return {
        'input'  : 2 * cm * cn * ck / (cm * ck * B_IN),        # = cn  = min(N, NT)
        'weight' : 2 * cm * cn * ck / (ck * cn * B_W),         # = 4*cm = 4*min(M, MT)
        'scale'  : 2 * cm * cn * ck / ((ck/qblk) * cn * B_Q),  # = cm*qblk
        'zp'     : 2 * cm * cn * ck / ((ck/qblk) * cn * B_Q),
    }

In [ ]:
# ---- Workload OIs for typical shapes ----
# (cm, cn, ck) here are per-tile ceiling values = min(M, MT=128), min(N, NT=128), min(K, KT=128)
shapes = [
    # name,                          cm,   cn,   ck,  qblk, k_tiles
    ('FFN large (M,N>=128, K=4096)',  128, 128, 128, 32,  32),
    ('FFN K=128 (1 k-tile)',          128, 128, 128, 32,  1),
    ('small-M (M=32, N=128)',          32, 128, 128, 32,  32),
    ('small-N (M=128, N=32) <-- worst', 128, 32, 128, 32,  32),
    ('tiny (M=N=32)',                   32,  32, 128, 32,  32),
    ('qblk=128 variant',              128, 128, 128, 128, 32),
]
print(f"{'shape':<38} {'qblk':>4} {'k_tiles':>7} {'OI_HBM':>8} {'OI_TMEM':>8}")
print('-' * 70)
for name, cm, cn, ck, qblk, kt in shapes:
    print(f'{name:<38} {qblk:>4} {kt:>7} {oi_hbm(cm,cn,ck,qblk,k_tiles=kt):>8.2f} {oi_tmem(cm,cn,ck,qblk,kt):>8.2f}')

print()
print('Per-tensor HBM OI (large shape, cm=cn=ck=128, qblk=32):')
for k, v in per_tensor_oi_hbm(128, 128, 128, 32).items():
    print(f'  {k:<8} {v:>7.1f} FLOPs/B')
print('Per-tensor HBM OI (small-N, cm=128,cn=32,ck=128, qblk=32):')
for k, v in per_tensor_oi_hbm(128, 32, 128, 32).items():
    print(f'  {k:<8} {v:>7.1f} FLOPs/B   <-- input OI = cn = 32 is the bottleneck')

## 2. Baseline roofline (current arch: HP=8, TB=8)

In [ ]:
OI_HBM_FFN   = oi_hbm(128, 128, 128, 32, k_tiles=32)
OI_TMEM_FFN  = oi_tmem(128, 128, 128, 32, k_tiles=32)
OI_HBM_SMN   = oi_hbm(128,  32, 128, 32, k_tiles=32)   # small-N
OI_TMEM_SMN  = oi_tmem(128, 32, 128, 32, k_tiles=32)
OI_HBM_SMM   = oi_hbm( 32, 128, 128, 32, k_tiles=32)   # small-M
OI_TMEM_SMM  = oi_tmem(32, 128, 128, 32, k_tiles=32)

def build_model(HP, TB, name_suffix=''):
    m = RooflineModel()
    m.add_compute(f'MXU 32x32 ({MXU_GOPS:.0f} GOps/s)', MXU_GOPS)
    m.add_bw(f'HBM (HP={HP}) {hbm_gbps(HP):.1f} GB/s', hbm_gbps(HP))
    m.add_bw(f'TMEM (TB={TB}) {tmem_gbps(TB):.1f} GB/s', tmem_gbps(TB), linestyle='--')
    m.add_bw(f'MXU input port ({hbm_gbps(1):.1f} GB/s)', hbm_gbps(1), linestyle=':')
    m.add_workload(f'FFN large @ HBM{name_suffix}',   oi=OI_HBM_FFN,  bws=[f'HBM (HP={HP}) {hbm_gbps(HP):.1f} GB/s'])
    m.add_workload(f'FFN large @ TMEM{name_suffix}',  oi=OI_TMEM_FFN, bws=[f'TMEM (TB={TB}) {tmem_gbps(TB):.1f} GB/s'])
    m.add_workload(f'small-N (M=128,N=32) @ HBM',     oi=OI_HBM_SMN,  bws=[f'HBM (HP={HP}) {hbm_gbps(HP):.1f} GB/s'])
    m.add_workload(f'small-N @ TMEM',                 oi=OI_TMEM_SMN, bws=[f'TMEM (TB={TB}) {tmem_gbps(TB):.1f} GB/s'])
    m.add_workload(f'small-M (M=32,N=128) @ HBM',     oi=OI_HBM_SMM,  bws=[f'HBM (HP={HP}) {hbm_gbps(HP):.1f} GB/s'])
    return m

m = build_model(HP=8, TB=8)
m.plot(title=f'Baseline roofline (HP=8, TB=8, F={F_GHZ*1000:.0f} MHz)', oi_range=(0.5, 4096))
m.summary()

## 3. HP sweep (TMEM fixed TB=8)

HBM port count 에 따라 HBM ridge 가 어떻게 움직이는지. 
Large shape (OI_HBM ≈ 97.5) 는 HP=1 에서도 compute-bound 지만, 
**small-N shape (OI_HBM ≈ 29.5)** 는 HP=1 ridge=32 아래로 내려가 memory-bound 로 빠짐.

In [ ]:
HP_SWEEP = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(HP_SWEEP), figsize=(18, 5), sharey=True)
for ax, hp in zip(axes, HP_SWEEP):
    mm = RooflineModel()
    mm.add_compute(f'MXU ({MXU_GOPS:.0f})', MXU_GOPS)
    mm.add_bw(f'HBM HP={hp}', hbm_gbps(hp))
    mm.add_bw(f'TMEM TB=8', tmem_gbps(8), linestyle='--')
    mm.add_workload('FFN large',   oi=OI_HBM_FFN,  bws=[f'HBM HP={hp}'])
    mm.add_workload('small-N',     oi=OI_HBM_SMN,  bws=[f'HBM HP={hp}'])
    mm.add_workload('small-M',     oi=OI_HBM_SMM,  bws=[f'HBM HP={hp}'])
    mm.plot(ax=ax, oi_range=(0.5, 4096), title=f'HP={hp}  (ridge={MXU_GOPS/hbm_gbps(hp):.1f})',
            show_ridge=False, show_annotations=True,
            workload_fontsize=9, ceiling_fontsize=8)
plt.suptitle(f'HP sweep, TB=8 fixed (F={F_GHZ*1000:.0f} MHz)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. TB sweep (HBM fixed HP=1)

TMEM bank 수 에 따라 TMEM ridge 가 움직임.
Large shape OI_TMEM ≈ 22.5 이므로 TB>=2 compute-bound.
**small-N OI_TMEM ≈ 14.7** 이면 TB=2 (ridge=16) 도 아직 memory-bound, TB>=3 필요.

In [ ]:
TB_SWEEP = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(TB_SWEEP), figsize=(18, 5), sharey=True)
for ax, tb in zip(axes, TB_SWEEP):
    mm = RooflineModel()
    mm.add_compute(f'MXU ({MXU_GOPS:.0f})', MXU_GOPS)
    mm.add_bw(f'HBM HP=1', hbm_gbps(1))
    mm.add_bw(f'TMEM TB={tb}', tmem_gbps(tb), linestyle='--')
    mm.add_workload('FFN @ HBM',      oi=OI_HBM_FFN,  bws=[f'HBM HP=1'])
    mm.add_workload('FFN @ TMEM',     oi=OI_TMEM_FFN, bws=[f'TMEM TB={tb}'])
    mm.add_workload('small-N @ TMEM', oi=OI_TMEM_SMN, bws=[f'TMEM TB={tb}'])
    mm.plot(ax=ax, oi_range=(0.5, 4096), title=f'TB={tb}  (ridge={MXU_GOPS/tmem_gbps(tb):.1f})',
            show_ridge=False, show_annotations=True,
            workload_fontsize=9, ceiling_fontsize=8)
plt.suptitle(f'TB sweep, HP=1 fixed (F={F_GHZ*1000:.0f} MHz)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. HP x TB heatmap - attainable MXU utilization

HBM 과 TMEM 의 BW bound 를 동시에 고려해, 각 (HP, TB) 조합에서 MXU peak 대비 달성 가능한 %
를 두 shape (large FFN, small-N) 각각에 대해 plot.

In [ ]:
import numpy as np

HP_list = [1, 2, 4, 8]
TB_list = [1, 2, 3, 4, 6, 8]

def attainable(HP, TB, oi_hbm_val, oi_tmem_val):
    bw_hbm_bound  = hbm_gbps(HP)  * oi_hbm_val
    bw_tmem_bound = tmem_gbps(TB) * oi_tmem_val
    return min(MXU_GOPS, bw_hbm_bound, bw_tmem_bound)

def heatmap_ax(ax, oi_hbm_val, oi_tmem_val, title):
    Z = np.zeros((len(HP_list), len(TB_list)))
    for i, hp in enumerate(HP_list):
        for j, tb in enumerate(TB_list):
            Z[i, j] = 100 * attainable(hp, tb, oi_hbm_val, oi_tmem_val) / MXU_GOPS
    im = ax.imshow(Z, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
    ax.set_xticks(range(len(TB_list))); ax.set_xticklabels([f'{t}' for t in TB_list])
    ax.set_yticks(range(len(HP_list))); ax.set_yticklabels([f'{h}' for h in HP_list])
    ax.set_xlabel('TB (TMEM banks)'); ax.set_ylabel('HP (HBM ports)')
    ax.set_title(f'{title}\n(OI_HBM={oi_hbm_val:.1f}, OI_TMEM={oi_tmem_val:.1f})')
    for i in range(len(HP_list)):
        for j in range(len(TB_list)):
            ax.text(j, i, f'{Z[i,j]:.0f}%', ha='center', va='center',
                    color='white' if Z[i,j] < 50 else 'black', fontweight='bold')
    return im

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
im0 = heatmap_ax(axes[0], OI_HBM_FFN, OI_TMEM_FFN, 'FFN large (M=N=128)')
im1 = heatmap_ax(axes[1], OI_HBM_SMN, OI_TMEM_SMN, 'small-N (M=128, N=32)')
fig.colorbar(im1, ax=axes.ravel().tolist(), label='% of MXU peak', shrink=0.8)
plt.suptitle('Attainable MXU utilization vs (HP, TB)', fontsize=13, fontweight='bold')
plt.show()

for label, oi_h, oi_t in [('FFN large', OI_HBM_FFN, OI_TMEM_FFN),
                          ('small-N',   OI_HBM_SMN, OI_TMEM_SMN)]:
    print(f'\n--- {label} (OI_HBM={oi_h:.1f}, OI_TMEM={oi_t:.1f}) ---')
    print('      ' + ''.join(f'TB={tb:<6}' for tb in TB_list))
    for hp in HP_list:
        row = f'HP={hp}: '
        for tb in TB_list:
            a = attainable(hp, tb, oi_h, oi_t)
            pct = 100 * a / MXU_GOPS
            row += f'{pct:5.1f}%  '
        print(row)

## 6. Interactive dashboard (optional)

ipywidgets 설치되어 있으면 slider 로 HP/TB/OI 를 직접 움직여볼 수 있다.

In [ ]:
try:
    import ipywidgets  # noqa: F401
    m_int = build_model(HP=4, TB=4, name_suffix='')
    m_int.interactive(figsize=(11, 5), oi_range=(0.5, 4096))
except ImportError:
    print('ipywidgets not installed - skipping interactive dashboard.')

## Conclusion

- Per-HBM-load reuse: **input=NT=128, weight=MT=128, scale=MT*qblk=4096**
  (not N/M/M*qblk; kernel re-fetches input per nt_dma, weight/scale per mt).
- **Large-shape (M,N ≥ 128)**: OI_HBM ≈ 97.5, OI_TMEM ≈ 22.5
  - HBM: **HP=1 이미 compute-bound**. HP=8 은 8×+ 과잉.
  - TMEM: **TB≥2 compute-bound**, TB=3~4 가 안전한 sweet spot.
- **small-N corner case (N < NT)** 가 가장 tight: OI_input = N 으로 떨어짐.
  - N=32: OI_HBM ≈ 29.5 → HP=1 ridge=32 아래 → HP≥2 필요.
  - N=32: OI_TMEM ≈ 14.7 → TB=2 ridge=16 아래 → TB≥3 필요.
- **small-M (M < MT)** 는 weight/scale OI 감소하지만 전체 OI 가 input (=128) 에서 saturated → HBM HP=1, TMEM TB=2 로도 OK.
- **Kernel-side 개선 여지**: nt_dma 루프에서 input 을 TMEM 에 pin 하여 재-load 제거하면
  input 의 per-load reuse 가 NT → N 까지 확장 → large-N shape 에서 OI_HBM 이 더욱 개선됨.